In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q1-ka-ai-2026")

print("Path to dataset files:", path)

In [ ]:
# Task 1: Write your code here:
import pandas as pd
import os

data_path = os.path.join(path, "Q1_data.csv")
# Load the dataset
df_FDT = pd.read_csv(data_path)

print(f"Dataset shape: {df_FDT.shape}")

In [ ]:
# Task 2: Write your code here:

df_FDT.head()

In [ ]:
# Task 3: Write your code here:

df_FDT.info()

In [ ]:
# Task 4: Write your code here:
df_FDT.describe()

In [ ]:
# Task 5: Write your code here:
import matplotlib.pyplot as plt

# Price distribution (target variable)
plt.figure(figsize=(10, 5))
plt.hist(df_FDT['Delivery_Time'], bins=50, edgecolor='black')
plt.title('Target Column')
plt.xlabel('Time')
plt.ylabel('Frequency')
plt.show()

In [ ]:
# Task 1: Write your code here:
df_FDT = df_FDT.drop(['Order_ID'], axis=1)
df_FDT.head()

In [ ]:
# Task 2: Write your code here:
df_FDT.isnull().sum()

df_FDT_2 = df_FDT.dropna(subset=['Weather', 'Traffic_Level', 'Time_of_Day', 'Courier_Experience_yrs', 'Delivery_Time'])
df_FDT_2.isnull().sum()

In [ ]:
# Task 3: Write your code here:
def check_duplicates(df):
  duplicates = df.duplicated().sum()
  print(f"Number of Duplicate Samples: {duplicates}")
  if duplicates > 0:
    print("Dropping Duplicates...")
    df.drop_duplicates(inplace=True)
    print("Duplicates Dropped.")
  else:
    print("No Duplicate Samples Found.")

check_duplicates(df_FDT_2)
df_FDT_2.shape

In [ ]:
# Task 4: Write your code here:
from sklearn.preprocessing import LabelEncoder

categorical_cols = df_FDT_2.select_dtypes(include=["object"]).columns
for col in categorical_cols:
    print(f"Encoding column: {col}")
    le = LabelEncoder()
    df_FDT_2[col] = le.fit_transform(df_FDT_2[col])
df_FDT_2

In [ ]:
# Task 5: Write your code here:
from sklearn.preprocessing import StandardScaler

numerical_cols = df_FDT_2.select_dtypes(include=["int64", "float64"]).columns.drop("Delivery_Time")  ### DON'T SCALE THE TARGET !!!!!!!!!!!!
scaler = StandardScaler()
df_FDT_2[numerical_cols] = scaler.fit_transform(df_FDT_2[numerical_cols])
df_FDT_2

In [ ]:
# Task 6: Write your code here:

# NO NEED!

In [ ]:
# Task 1: Write your code here:
# Prepare the data as X and y

X = df_FDT_2.drop("Delivery_Time", axis=1)
y = df_FDT_2["Delivery_Time"]

X.head()

In [ ]:
y.head()

In [ ]:
# Task 2,3,4,5: Write your code here:
from sklearn.model_selection import KFold
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error as m_MAE

n_splits = 5

# Define K-Fold Cross Validation
kf = KFold(n_splits=5, shuffle=True, random_state=42)



# Iterate through folds
for fold, (train_idx, test_idx) in enumerate(kf.split(X), start=1):
    # indexing for each fold
    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
    # print shapes
    print(f"Fold {fold}")
    print("  X_train shape:", X_train.shape)
    print("  X_test shape :", X_test.shape)
    print("  y_train shape:", y_train.shape)
    print("  y_test shape :", y_test.shape)
    print("-" * 30)

models = {
  "Random Forest Regressor": RandomForestRegressor(n_estimators=200)
}

# Storage for results
all_results = {}

for name in models:
  all_results[name] = {'mae': []}



kf = KFold(n_splits=5, shuffle=True, random_state=42)

for fold_idx, (train_index, test_index) in enumerate(kf.split(X)):
  print(f"\nFold {fold_idx + 1}/{n_splits}")

  X_train, X_test = X.iloc[train_index], X.iloc[test_index]
  y_train, y_test = y.iloc[train_index], y.iloc[test_index]

  for model_name, model in models.items():
    print(f"Training {model_name}...")

    # Train
    model.fit(X_train, y_train)

    # Predict
    y_pred = model.predict(X_test)

    # Calculate metrics
    mae = m_MAE(y_test, y_pred)

    # Store results
    all_results[model_name]["mae"].append(mae)

In [ ]:
import numpy as np
print(f"  MAE:  {np.mean(mae):.4f}")

In [ ]:
baseline_pred = np.full_like(y, y.mean())

baseline_mae = m_MAE(y, baseline_pred)
print(f"Baseline MSE (using mean target): {baseline_mae:.4f}")

In [ ]:
# Task 1: Write your code here:
coeffs = {}

coeffs['rfr'] = models['Random Forest Regressor'].coef_

fig, axes = plt.subplots(1, 2, figsize=(15, 6))
axes = axes.flatten()
features = X.columns

for i, (model_name, coef) in enumerate(coeffs.items()):
  # Sort features by absolute coefficient value
  absolute_coef = np.abs(coef)
  sorted_idx = np.argsort(absolute_coef)

  ax = axes[i]
  ax.barh(features[sorted_idx], coef[sorted_idx])
  ax.set_title(f"{model_name} Coefficients")
  ax.set_xlabel("Coefficient Value (Impact)")

plt.tight_layout()
plt.show()

In [ ]:
# Task 2: Write your code here:

In [ ]:
# Task Bonus: Write your code here: